In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

# Heijden 2021 data from the SAS file
raw = pd.DataFrame([
    [1, 1, 1, 0, 0, 3004335],
    [2, 1, 1, 0, 1, 31995],
    [3, 1, 1, 0, np.nan, 150840],
    [4, 1, 0, 0, np.nan, 38634],
    [5, 1, 1, 1, 0, 108189],
    [6, 1, 1, 1, 1, 435465],
    [7, 1, 1, 1, np.nan, 12405],
    [8, 1, 0, 1, np.nan, 4368],
    [9, 1, 1, np.nan, 0, 16512],
    [10, 1, 1, np.nan, 1, 2769],
    [11, 1, 1, np.nan, np.nan, 900],
    [12, 1, 0, np.nan, np.nan, 438],
    [13, 0, 1, np.nan, 0, 398838],
    [14, 0, 1, np.nan, 1, 146976],
    [15, 0, 1, np.nan, np.nan, 24636],
    [16, 0, 0, np.nan, np.nan, np.nan],
], columns=["id", "list1", "list2", "class1", "class2", "count"])


def expand_uniform(df):
    class1_vals = sorted(df["class1"].dropna().unique())
    class2_vals = sorted(df["class2"].dropna().unique())

    rows = []
    for _, r in df.iterrows():
        c1s = class1_vals if pd.isna(r.class1) else [r.class1]
        c2s = class2_vals if pd.isna(r.class2) else [r.class2]

        denom = len(c1s) * len(c2s)
        for c1 in c1s:
            for c2 in c2s:
                rr = r.copy()
                rr["class1"] = c1
                rr["class2"] = c2
                rr["count_init"] = r["count"] / denom if pd.notna(r["count"]) else np.nan
                rows.append(rr)

    return pd.DataFrame(rows)


def fit_expected(df, response):
    # SAS: model = list1|class2 class1|class2 list2|class1
    formula = (
        f"{response} ~ "
        "C(list1) * C(class2) + "
        "C(class1) * C(class2) + "
        "C(list2) * C(class1)"
    )

    train = df[df[response].notna()].copy()

    model = smf.glm(
        formula=formula,
        data=train,
        family=sm.families.Poisson()
    ).fit(maxiter=100)

    return model.predict(df)


def reallocate_within_id(df, expectation_col, source_col):
    out = df.copy()

    exp_tot = out.groupby("id")[expectation_col].transform("sum")
    org_tot = out.groupby("id")[source_col].transform("sum")

    adjusted = (out[expectation_col] / exp_tot) * org_tot

    # SAS excludes list1=0 and list2=0 hidden cell from tmp during iteration
    hidden = (out["list1"] == 0) & (out["list2"] == 0)

    out[f"{expectation_col}_tmp"] = np.where(
        hidden,
        np.nan,
        np.round(adjusted)
    )

    return out


def vdh_em(raw, max_iter=1000, tol=1e-4):
    df = expand_uniform(raw)

    prev_expect = None

    # iteration 1
    df["expectation_1"] = fit_expected(df, "count_init")
    df = reallocate_within_id(df, "expectation_1", "count_init")

    mse = np.inf
    iteration = 1

    while iteration < max_iter and mse > tol:
        prev = iteration
        iteration += 1

        response = f"expectation_{prev}_tmp"
        pred_col = f"expectation_{iteration}"

        df[pred_col] = fit_expected(df, response)
        df = reallocate_within_id(df, pred_col, "count_init")

        mse = np.nanmean((df[pred_col] - df[f"expectation_{prev}"]) ** 2)

    # final allocation, including hidden cell
    final_col = f"expectation_{iteration}"
    exp_tot = df.groupby("id")[final_col].transform("sum")
    org_tot = df.groupby("id")["count_init"].transform("sum")
    df["final"] = np.round((df[final_col] / exp_tot) * org_tot)

    return df, iteration, mse


result, n_iter, mse = vdh_em(raw)

print("iterations:", n_iter)
print("mse:", mse)

print(
    result[
        ["id", "list1", "list2", "class1", "class2", "count_init", "final"]
    ].sort_values(["list1", "class1", "list2", "class2"], ascending=False)
)

iterations: 12
mse: 0.0
      id  list1  list2  class1  class2  count_init      final
5    6.0    1.0    1.0     1.0     1.0    435465.0   435465.0
6    7.0    1.0    1.0     1.0     1.0      6202.5     9938.0
9   10.0    1.0    1.0     1.0     1.0      1384.5     2575.0
10  11.0    1.0    1.0     1.0     1.0       225.0      107.0
4    5.0    1.0    1.0     1.0     0.0    108189.0   108189.0
6    7.0    1.0    1.0     1.0     0.0      6202.5     2467.0
8    9.0    1.0    1.0     1.0     0.0      8256.0      560.0
10  11.0    1.0    1.0     1.0     0.0       225.0       27.0
7    8.0    1.0    0.0     1.0     1.0      2184.0     3499.0
11  12.0    1.0    0.0     1.0     1.0       109.5       36.0
7    8.0    1.0    0.0     1.0     0.0      2184.0      869.0
11  12.0    1.0    0.0     1.0     0.0       109.5        9.0
1    2.0    1.0    1.0     0.0     1.0     31995.0    31995.0
2    3.0    1.0    1.0     0.0     1.0     75420.0     1591.0
9   10.0    1.0    1.0     0.0     1.0      13

In [2]:
import numpy as np
import pandas as pd

raw_2018 = pd.DataFrame([
    [1, "1", "1", "0", "0", 259],
    [2, "1", "1", "0", "1", 539],
    [3, "1", "1", "1", "0", 110],
    [4, "1", "1", "1", "1", 177],
    [5, "0", "1", np.nan, "0", 91],
    [6, "0", "1", np.nan, "1", 164],
    [7, "1", "0", "0", np.nan, 13898],
    [8, "1", "0", "1", np.nan, 12356],
    [9, "0", "0", np.nan, np.nan, np.nan],
], columns=["id", "list1", "list2", "class1", "class2", "count"])

result_2018, n_iter_2018, mse_2018 = vdh_em(raw_2018)

print("iterations:", n_iter_2018)
print("mse:", mse_2018)

print(
    result_2018[
        ["id", "list1", "list2", "class1", "class2", "count_init", "final"]
    ].sort_values(["list1", "class1", "list2", "class2"], ascending=False)
)

iterations: 196
mse: 0.0
   id list1 list2 class1 class2  count_init   final
3   4     1     1      1      1       177.0   177.0
2   3     1     1      1      0       110.0   110.0
7   8     1     0      1      1      6178.0  7563.0
7   8     1     0      1      0      6178.0  4793.0
1   2     1     1      0      1       539.0   539.0
0   1     1     1      0      0       259.0   259.0
6   7     1     0      0      1      6949.0  9346.0
6   7     1     0      0      0      6949.0  4552.0
5   6     0     1      1      1        82.0    40.0
4   5     0     1      1      0        45.5    27.0
8   9     0     0      1      1         NaN     0.0
8   9     0     0      1      0         NaN     0.0
5   6     0     1      0      1        82.0   124.0
4   5     0     1      0      0        45.5    64.0
8   9     0     0      0      1         NaN     0.0
8   9     0     0      0      0         NaN     0.0


In [3]:
tab = (
    result_2018
    .rename(columns={
        "list1": "A",
        "list2": "B",
        "class1": "X1",
        "class2": "X2",
    })
    .pivot_table(
        index=["A", "X1"],
        columns=["B", "X2"],
        values="final",
        aggfunc="sum"
    )
)

# Match the screenshot ordering
tab = tab.sort_index(
    level=["A", "X1"],
    ascending=[False, True]
)

tab = tab.sort_index(
    axis=1,
    level=["B", "X2"],
    ascending=[False, True]
)

print(tab.round(1))

B         1              0        
X2        0      1       0       1
A X1                              
1 0   259.0  539.0  4552.0  9346.0
  1   110.0  177.0  4793.0  7563.0
0 0    64.0  124.0     0.0     0.0
  1    27.0   40.0     0.0     0.0


In [4]:
tab.round(1).style.format("{:,.1f}")

In [5]:
tab_flat = tab.copy()
tab_flat.columns = [f"B={b}, X2={x2}" for b, x2 in tab_flat.columns]
tab_flat = tab_flat.reset_index()

print(tab_flat.round(1))

   A X1  B=1, X2=0  B=1, X2=1  B=0, X2=0  B=0, X2=1
0  1  0      259.0      539.0     4552.0     9346.0
1  1  1      110.0      177.0     4793.0     7563.0
2  0  0       64.0      124.0        0.0        0.0
3  0  1       27.0       40.0        0.0        0.0


In [6]:
def make_crosstab(
    df,
    value_col="final",
    row_vars=("list1", "class1"),
    col_vars=("list2", "class2"),
    row_names=("A", "X1"),
    col_names=("B", "X2"),
    decimals=1,
    flatten=False,
):
    d = df.copy()

    rename_map = {
        row_vars[0]: row_names[0],
        row_vars[1]: row_names[1],
        col_vars[0]: col_names[0],
        col_vars[1]: col_names[1],
    }
    d = d.rename(columns=rename_map)

    row_vars = list(row_names)
    col_vars = list(col_names)

    tab = d.pivot_table(
        index=row_vars,
        columns=col_vars,
        values=value_col,
        aggfunc="sum",
    )

    tab = tab.sort_index(
        level=row_vars,
        ascending=[False, True],
    )

    tab = tab.sort_index(
        axis=1,
        level=col_vars,
        ascending=[False, True],
    )

    tab = tab.round(decimals)

    if flatten:
        tab = tab.copy()
        tab.columns = [
            ", ".join(f"{name}={val}" for name, val in zip(col_names, col))
            for col in tab.columns
        ]
        tab = tab.reset_index()

    return tab

In [10]:
make_crosstab(result_2018, flatten=True)

,A,X1,"B=1, X2=0","B=1, X2=1","B=0, X2=0","B=0, X2=1"
0,1,0,259.0,539.0,4552.0,9346.0
1,1,1,110.0,177.0,4793.0,7563.0
2,0,0,64.0,124.0,0.0,0.0
3,0,1,27.0,40.0,0.0,0.0


In [12]:
make_crosstab(result, flatten=True)

,A,X1,"B=1.0, X2=0.0","B=1.0, X2=1.0","B=0.0, X2=0.0","B=0.0, X2=1.0"
0,1.0,0.0,3170294.0,33788.0,38616.0,411.0
1,1.0,1.0,111243.0,448085.0,878.0,3535.0
2,0.0,0.0,402709.0,10771.0,0.0,0.0
3,0.0,1.0,14131.0,142839.0,0.0,0.0


In [13]:
def style_crosstab(tab, decimals=1):
    numeric_cols = tab.select_dtypes(include="number").columns

    return tab.style.format(
        {col: f"{{:,.{decimals}f}}" for col in numeric_cols}
    )
tab_flat = make_crosstab(result, flatten=True)

style_crosstab(tab_flat, decimals=1)

,A,X1,"B=1.0, X2=0.0","B=1.0, X2=1.0","B=0.0, X2=0.0","B=0.0, X2=1.0"
0,1.0,0.0,"3,170,294.0","33,788.0","38,616.0",411.0
1,1.0,1.0,"111,243.0","448,085.0",878.0,"3,535.0"
2,0.0,0.0,"402,709.0","10,771.0",0.0,0.0
3,0.0,1.0,"14,131.0","142,839.0",0.0,0.0


In [14]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm


def expand_missing_classes(df, class1_col="class1", class2_col="class2", count_col="count"):
    class1_vals = sorted(df[class1_col].dropna().unique())
    class2_vals = sorted(df[class2_col].dropna().unique())

    rows = []

    for _, r in df.iterrows():
        c1s = class1_vals if pd.isna(r[class1_col]) else [r[class1_col]]
        c2s = class2_vals if pd.isna(r[class2_col]) else [r[class2_col]]

        denom = len(c1s) * len(c2s)

        for c1 in c1s:
            for c2 in c2s:
                rr = r.copy()
                rr[class1_col] = c1
                rr[class2_col] = c2

                if pd.notna(r[count_col]):
                    rr["count_init"] = r[count_col] / denom
                else:
                    rr["count_init"] = np.nan

                rows.append(rr)

    return pd.DataFrame(rows)


def fit_poisson_loglinear(
    df,
    response_col,
    list1_col="list1",
    list2_col="list2",
    class1_col="class1",
    class2_col="class2",
):
    formula = (
        f"{response_col} ~ "
        f"C({list1_col}) * C({class2_col}) + "
        f"C({class1_col}) * C({class2_col}) + "
        f"C({list2_col}) * C({class1_col})"
    )

    train = df[df[response_col].notna()].copy()

    model = smf.glm(
        formula=formula,
        data=train,
        family=sm.families.Poisson()
    ).fit(maxiter=100)

    return model.predict(df)


def reallocate_observed_cells(
    df,
    expectation_col,
    source_col="count_init",
    id_col="id",
    list1_col="list1",
    list2_col="list2",
):
    out = df.copy()

    hidden = (out[list1_col].astype(str) == "0") & (out[list2_col].astype(str) == "0")

    exp_tot = out.groupby(id_col)[expectation_col].transform("sum")
    org_tot = out.groupby(id_col)[source_col].transform(lambda x: x.sum(min_count=1))

    out[f"{expectation_col}_tmp"] = np.where(
        hidden,
        np.nan,
        (out[expectation_col] / exp_tot) * org_tot
    )

    return out


def finalize_estimates(
    df,
    expectation_col,
    source_col="count_init",
    id_col="id",
    list1_col="list1",
    list2_col="list2",
    decimals=1,
):
    out = df.copy()

    hidden = (out[list1_col].astype(str) == "0") & (out[list2_col].astype(str) == "0")

    exp_tot = out.groupby(id_col)[expectation_col].transform("sum")
    org_tot = out.groupby(id_col)[source_col].transform(lambda x: x.sum(min_count=1))

    out["estimate"] = np.where(
        hidden,
        out[expectation_col],                      # dual-system estimate
        (out[expectation_col] / exp_tot) * org_tot # observed/missing-class allocation
    )

    out["estimate"] = out["estimate"].round(decimals)

    return out


def vdh_em_estimator(
    raw,
    id_col="id",
    list1_col="list1",
    list2_col="list2",
    class1_col="class1",
    class2_col="class2",
    count_col="count",
    max_iter=1000,
    tol=1e-4,
    decimals=1,
):
    df = expand_missing_classes(
        raw,
        class1_col=class1_col,
        class2_col=class2_col,
        count_col=count_col,
    )

    iteration = 1

    df[f"expectation_{iteration}"] = fit_poisson_loglinear(
        df,
        response_col="count_init",
        list1_col=list1_col,
        list2_col=list2_col,
        class1_col=class1_col,
        class2_col=class2_col,
    )

    df = reallocate_observed_cells(
        df,
        expectation_col=f"expectation_{iteration}",
        source_col="count_init",
        id_col=id_col,
        list1_col=list1_col,
        list2_col=list2_col,
    )

    mse = np.inf

    while iteration < max_iter and mse > tol:
        prev = iteration
        iteration += 1

        response_col = f"expectation_{prev}_tmp"
        expectation_col = f"expectation_{iteration}"

        df[expectation_col] = fit_poisson_loglinear(
            df,
            response_col=response_col,
            list1_col=list1_col,
            list2_col=list2_col,
            class1_col=class1_col,
            class2_col=class2_col,
        )

        df = reallocate_observed_cells(
            df,
            expectation_col=expectation_col,
            source_col="count_init",
            id_col=id_col,
            list1_col=list1_col,
            list2_col=list2_col,
        )

        mse = np.nanmean(
            (df[expectation_col] - df[f"expectation_{prev}"]) ** 2
        )

    final_col = f"expectation_{iteration}"

    df = finalize_estimates(
        df,
        expectation_col=final_col,
        source_col="count_init",
        id_col=id_col,
        list1_col=list1_col,
        list2_col=list2_col,
        decimals=decimals,
    )

    return df, {
        "iterations": iteration,
        "mse": mse,
        "final_expectation_col": final_col,
    }

In [15]:
raw_2018 = pd.DataFrame([
    [1, 1, 1, 0, 0, 259],
    [2, 1, 1, 0, 1, 539],
    [3, 1, 1, 1, 0, 110],
    [4, 1, 1, 1, 1, 177],
    [5, 0, 1, np.nan, 0, 91],
    [6, 0, 1, np.nan, 1, 164],
    [7, 1, 0, 0, np.nan, 13898],
    [8, 1, 0, 1, np.nan, 12356],
    [9, 0, 0, np.nan, np.nan, np.nan],
], columns=["id", "list1", "list2", "class1", "class2", "count"])

est_2018, info_2018 = vdh_em_estimator(raw_2018)

print(info_2018)

est_2018[
    ["id", "list1", "list2", "class1", "class2", "count_init", "estimate"]
]

{'iterations': 310, 'mse': np.float64(9.685436961524537e-05), 'final_expectation_col': 'expectation_310'}


,id,list1,list2,class1,class2,count_init,estimate
0,1.0,1.0,1.0,0.0,0.0,259.0,259.0
1,2.0,1.0,1.0,0.0,1.0,539.0,539.0
2,3.0,1.0,1.0,1.0,0.0,110.0,110.0
3,4.0,1.0,1.0,1.0,1.0,177.0,177.0
4,5.0,0.0,1.0,0.0,0.0,45.5,63.9
4,5.0,0.0,1.0,1.0,0.0,45.5,27.1
5,6.0,0.0,1.0,0.0,1.0,82.0,123.5
5,6.0,0.0,1.0,1.0,1.0,82.0,40.5
6,7.0,1.0,0.0,0.0,0.0,6949.0,4510.8
6,7.0,1.0,0.0,0.0,1.0,6949.0,9387.2


In [16]:
def estimate_crosstab(
    df,
    value_col="estimate",
    row_vars=("list1", "class1"),
    col_vars=("list2", "class2"),
    row_names=("A", "X1"),
    col_names=("B", "X2"),
    decimals=1,
    flatten=False,
):
    d = df.rename(columns={
        row_vars[0]: row_names[0],
        row_vars[1]: row_names[1],
        col_vars[0]: col_names[0],
        col_vars[1]: col_names[1],
    })

    tab = d.pivot_table(
        index=list(row_names),
        columns=list(col_names),
        values=value_col,
        aggfunc=lambda x: x.sum(min_count=1),
    )

    tab = tab.sort_index(level=list(row_names), ascending=[False, True])
    tab = tab.sort_index(axis=1, level=list(col_names), ascending=[False, True])
    tab = tab.round(decimals)

    if flatten:
        tab = tab.copy()
        tab.columns = [
            ", ".join(f"{name}={val}" for name, val in zip(col_names, col))
            for col in tab.columns
        ]
        tab = tab.reset_index()

    return tab


estimate_crosstab(est_2018)

B          1.0            0.0        
X2         0.0    1.0     0.0     1.0
A   X1                               
1.0 0.0  259.0  539.0  4510.8  9387.2
    1.0  110.0  177.0  4736.9  7619.1
0.0 0.0   63.9  123.5     NaN     NaN
    1.0   27.1   40.5     NaN     NaN

In [17]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm


def is_zero_col(s):
    return pd.to_numeric(s, errors="coerce").eq(0)


def expand_missing_classes(df, class1_col="class1", class2_col="class2", count_col="count"):
    class1_vals = sorted(df[class1_col].dropna().unique())
    class2_vals = sorted(df[class2_col].dropna().unique())

    rows = []

    for _, r in df.iterrows():
        c1s = class1_vals if pd.isna(r[class1_col]) else [r[class1_col]]
        c2s = class2_vals if pd.isna(r[class2_col]) else [r[class2_col]]

        denom = len(c1s) * len(c2s)

        for c1 in c1s:
            for c2 in c2s:
                rr = r.copy()
                rr[class1_col] = c1
                rr[class2_col] = c2
                rr["count_init"] = r[count_col] / denom if pd.notna(r[count_col]) else np.nan
                rows.append(rr)

    return pd.DataFrame(rows)


def fit_poisson_loglinear(
    df,
    response_col,
    list1_col="list1",
    list2_col="list2",
    class1_col="class1",
    class2_col="class2",
):
    formula = (
        f"{response_col} ~ "
        f"C({list1_col}) * C({class2_col}) + "
        f"C({class1_col}) * C({class2_col}) + "
        f"C({list2_col}) * C({class1_col})"
    )

    train = df[df[response_col].notna()].copy()

    model = smf.glm(
        formula=formula,
        data=train,
        family=sm.families.Poisson()
    ).fit(maxiter=100)

    return model.predict(df)


def reallocate_observed_cells(
    df,
    expectation_col,
    source_col="count_init",
    id_col="id",
    list1_col="list1",
    list2_col="list2",
):
    out = df.copy()

    hidden = is_zero_col(out[list1_col]) & is_zero_col(out[list2_col])

    exp_tot = out.groupby(id_col)[expectation_col].transform("sum")
    org_tot = out.groupby(id_col)[source_col].transform(lambda x: x.sum(min_count=1))

    out[f"{expectation_col}_tmp"] = np.where(
        hidden,
        np.nan,
        (out[expectation_col] / exp_tot) * org_tot
    )

    return out


def finalize_estimates(
    df,
    expectation_col,
    source_col="count_init",
    id_col="id",
    list1_col="list1",
    list2_col="list2",
    decimals=1,
):
    out = df.copy()

    hidden = is_zero_col(out[list1_col]) & is_zero_col(out[list2_col])

    exp_tot = out.groupby(id_col)[expectation_col].transform("sum")
    org_tot = out.groupby(id_col)[source_col].transform(lambda x: x.sum(min_count=1))

    out["estimate"] = np.where(
        hidden,
        out[expectation_col],                       # keep model estimate for hidden cells
        (out[expectation_col] / exp_tot) * org_tot  # allocate observed totals
    )

    out["estimate"] = out["estimate"].round(decimals)

    return out


def vdh_em_estimator(
    raw,
    id_col="id",
    list1_col="list1",
    list2_col="list2",
    class1_col="class1",
    class2_col="class2",
    count_col="count",
    max_iter=1000,
    tol=1e-4,
    decimals=1,
):
    df = expand_missing_classes(
        raw,
        class1_col=class1_col,
        class2_col=class2_col,
        count_col=count_col,
    )

    iteration = 1

    df[f"expectation_{iteration}"] = fit_poisson_loglinear(
        df,
        response_col="count_init",
        list1_col=list1_col,
        list2_col=list2_col,
        class1_col=class1_col,
        class2_col=class2_col,
    )

    df = reallocate_observed_cells(
        df,
        expectation_col=f"expectation_{iteration}",
        source_col="count_init",
        id_col=id_col,
        list1_col=list1_col,
        list2_col=list2_col,
    )

    mse = np.inf

    while iteration < max_iter and mse > tol:
        prev = iteration
        iteration += 1

        response_col = f"expectation_{prev}_tmp"
        expectation_col = f"expectation_{iteration}"

        df[expectation_col] = fit_poisson_loglinear(
            df,
            response_col=response_col,
            list1_col=list1_col,
            list2_col=list2_col,
            class1_col=class1_col,
            class2_col=class2_col,
        )

        df = reallocate_observed_cells(
            df,
            expectation_col=expectation_col,
            source_col="count_init",
            id_col=id_col,
            list1_col=list1_col,
            list2_col=list2_col,
        )

        mse = np.nanmean(
            (df[expectation_col] - df[f"expectation_{prev}"]) ** 2
        )

    final_col = f"expectation_{iteration}"

    df = finalize_estimates(
        df,
        expectation_col=final_col,
        source_col="count_init",
        id_col=id_col,
        list1_col=list1_col,
        list2_col=list2_col,
        decimals=decimals,
    )

    return df, {
        "iterations": iteration,
        "mse": mse,
        "final_expectation_col": final_col,
    }


def estimate_crosstab(
    df,
    value_col="estimate",
    row_vars=("list1", "class1"),
    col_vars=("list2", "class2"),
    row_names=("A", "X1"),
    col_names=("B", "X2"),
    decimals=1,
    flatten=False,
):
    d = df.rename(columns={
        row_vars[0]: row_names[0],
        row_vars[1]: row_names[1],
        col_vars[0]: col_names[0],
        col_vars[1]: col_names[1],
    })

    tab = d.pivot_table(
        index=list(row_names),
        columns=list(col_names),
        values=value_col,
        aggfunc=lambda x: x.sum(min_count=1),
    )

    tab = tab.sort_index(level=list(row_names), ascending=[False, True])
    tab = tab.sort_index(axis=1, level=list(col_names), ascending=[False, True])
    tab = tab.round(decimals)

    if flatten:
        tab = tab.copy()
        tab.columns = [
            ", ".join(f"{name}={val}" for name, val in zip(col_names, col))
            for col in tab.columns
        ]
        tab = tab.reset_index()

    return tab


def style_crosstab(tab, decimals=1):
    numeric_cols = tab.select_dtypes(include="number").columns
    return tab.style.format({col: f"{{:,.{decimals}f}}" for col in numeric_cols})


# -----------------------
# Example: 2018 data
# -----------------------

raw_2018 = pd.DataFrame([
    [1, 1, 1, 0, 0, 259],
    [2, 1, 1, 0, 1, 539],
    [3, 1, 1, 1, 0, 110],
    [4, 1, 1, 1, 1, 177],
    [5, 0, 1, np.nan, 0, 91],
    [6, 0, 1, np.nan, 1, 164],
    [7, 1, 0, 0, np.nan, 13898],
    [8, 1, 0, 1, np.nan, 12356],
    [9, 0, 0, np.nan, np.nan, np.nan],
], columns=["id", "list1", "list2", "class1", "class2", "count"])


est_2018, info_2018 = vdh_em_estimator(raw_2018)

print(info_2018)

print(
    est_2018[
        ["id", "list1", "list2", "class1", "class2", "count_init", "estimate"]
    ]
)

tab = estimate_crosstab(est_2018)
print(tab)

tab_flat = estimate_crosstab(est_2018, flatten=True)
print(tab_flat)

# In Jupyter:
# style_crosstab(tab)
# style_crosstab(tab_flat)

{'iterations': 310, 'mse': np.float64(9.685436961524537e-05), 'final_expectation_col': 'expectation_310'}
    id  list1  list2  class1  class2  count_init  estimate
0  1.0    1.0    1.0     0.0     0.0       259.0     259.0
1  2.0    1.0    1.0     0.0     1.0       539.0     539.0
2  3.0    1.0    1.0     1.0     0.0       110.0     110.0
3  4.0    1.0    1.0     1.0     1.0       177.0     177.0
4  5.0    0.0    1.0     0.0     0.0        45.5      63.9
4  5.0    0.0    1.0     1.0     0.0        45.5      27.1
5  6.0    0.0    1.0     0.0     1.0        82.0     123.5
5  6.0    0.0    1.0     1.0     1.0        82.0      40.5
6  7.0    1.0    0.0     0.0     0.0      6949.0    4510.8
6  7.0    1.0    0.0     0.0     1.0      6949.0    9387.2
7  8.0    1.0    0.0     1.0     0.0      6178.0    4736.9
7  8.0    1.0    0.0     1.0     1.0      6178.0    7619.1
8  9.0    0.0    0.0     0.0     0.0         NaN    1112.3
8  9.0    0.0    0.0     0.0     1.0         NaN    2150.2
8  9.0   